# 2. Conventional Approaches to Phase Identification

In [ ]:
# @title Environment Setup
!pip install pymatgen numpy matplotlib -q
# Installation should take > 1 minute to complete
# You may see a dependency conflict warning about 'requests'
# This is harmless and can be ignored
print("Packages installed")

In [ ]:
# @title Load Tutorial
import os

REPO = "/content/MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.isdir(REPO):
    %cd {REPO}
    !git fetch origin -q
    !git checkout main -q
    !git pull --ff-only -q
    print("Repo updated")
else:
    %cd /content
    !git clone {REPO_URL} -q
    %cd {REPO}
    print("Repo cloned")


## 2a) Search Match

Here we detect peaks in experimental patterns and rank candidate phases using FoM-style line matching scores.

In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

# We'll import some pre-written scripts from this repo
# You can find them in: MRS_CH08_Tutorial/tutorial_utils
import importlib
from tutorial_utils.conventional import search_match as sm
from tutorial_utils.sections import conventional_search_match as vis_sm

# Force reload to get latest changes
sm = importlib.reload(sm)
vis_sm = importlib.reload(vis_sm)


def _format_topk(rows, score_key, top_k):
    rows = [r for r in rows if r[score_key] > 0.0] or rows
    return ", ".join(
        f"{row['phase']} ({row[score_key]:.3f})" for row in rows[:top_k])


def run_search_match(
    top_k_to_print=3,             # phases to show per metric
    match_tolerance_deg=0.25,     # line-match window (2θ)
    min_peak_distance_deg=0.22,   # min spacing between detected peaks
    num_obs_lines_for_fom=20,     # strongest observed lines used
):

    # Set data paths (patterns, structures, and outputs)
    EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
    REFERENCE_DIR = Path("data/reference_structures")
    OUTPUT_DIR = Path("outputs/conventional/search_match")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MIN_ANGLE, MAX_ANGLE = 10.0, 80.0

    # Match plotting range to analysis range
    vis_sm.OUTPUT_DIR = OUTPUT_DIR
    vis_sm.TOP_K_TO_PRINT = top_k_to_print
    vis_sm.PLOT_MIN_ANGLE = MIN_ANGLE
    vis_sm.PLOT_MAX_ANGLE = MAX_ANGLE

    # Load experimental patterns
    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    # Step 0: build reference "stick" library
    refs = sm.load_reference_library(
        sorted(REFERENCE_DIR.glob("*.cif")),
        min_angle=MIN_ANGLE,
        max_angle=MAX_ANGLE,
    )

    all_rows = []
    print(f"Search-match | patterns={len(exp_files)} refs={len(refs)}")

    # Iterate through each pattern
    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: load and normalize the pattern
        tt, intensity = sm.load_pattern(exp_file,
            min_angle=MIN_ANGLE, max_angle=MAX_ANGLE)

        # Step 2: detect peaks
        _, obs_peaks = sm.detect_peaks(
            tt,
            intensity,
            min_peak_distance_deg=min_peak_distance_deg,
        )

        # Step 3: rank phases with FoM calculations
        by_dewolff, by_smith = sm.rank_phases(
            obs_peaks,
            refs,
            num_obs_lines_for_fom=num_obs_lines_for_fom,
            match_tolerance_deg=match_tolerance_deg,
        )

        print(f"\n{pattern_name} | peaks={len(obs_peaks)}")
        print(f"  de Wolff: {_format_topk(by_dewolff, 'de_wolff', top_k_to_print)}")
        print(f"  Smith-Snyder: {_format_topk(by_smith, 'smith_snyder', top_k_to_print)}")

        # Step 4: visualize the best matches
        vis_sm.plot_summary(
            pattern_name,
            tt,
            intensity,
            obs_peaks,
            by_dewolff[0],
            by_smith[0],
            refs,
            show_plot=True,
            display_size=900
        )


In [ ]:
# @title Run Search-Match
run_search_match()

# Try on your own:
  # Use fewer or more observed peaks in the FoM calculation.

# Examples below
  # run_search_match(num_obs_lines_for_fom=10)
  # run_search_match(num_obs_lines_for_fom=30)



## 2b) Profile Correlation


In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

import importlib
from tutorial_utils.conventional import profile_correlation as pc
from tutorial_utils.sections import conventional_profile_correlation as vis_pc

# Force reload to get latest changes
sm = importlib.reload(pc)
vis_sm = importlib.reload(vis_pc)

def _format_topk_profile(rows, score_key, top_k):
    return ", ".join(
        f"{row['phase']} ({row[score_key]:.3f})" for row in rows[:top_k])


def run_profile_correlation(
    top_k_to_print=3,          # phases to show per metric
    fwhm=0.30,                 # simulated peak width (deg)
    gauss_frac=0.2,            # Lorentzian fraction in profile mix
):

    # Set data paths (patterns, structures, and outputs)
    EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
    REFERENCE_DIR = Path("data/reference_structures")
    OUTPUT_DIR = Path("outputs/conventional/profile_correlation")
    MIN_ANGLE, MAX_ANGLE = 10.0, 80.0

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    vis_pc.OUTPUT_DIR = OUTPUT_DIR
    vis_pc.TOP_K_TO_PRINT = top_k_to_print
    vis_pc.FWHM = fwhm
    vis_pc.GAUSS_FRAC = gauss_frac

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    # Step 0: load reference "stick" patterns
    ref_lib = pc.load_reference_stick_library(
        sorted(REFERENCE_DIR.glob("*.cif")),
        min_angle=MIN_ANGLE,
        max_angle=MAX_ANGLE,
    )

    all_rows = []
    print(f"Profile correlation | patterns={len(exp_files)} refs={len(ref_lib)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: load and preprocess experimental patterns
        two_theta, exp_profile = pc.load_experimental_profile(
            exp_file,
            min_angle=MIN_ANGLE,
            max_angle=MAX_ANGLE,
        )

        # Step 2: simulate each candidate and compute similarity metrics
        by_pearson, by_cosine, simulated_profiles = pc.rank_phases(
            exp_profile,
            two_theta,
            ref_lib,
            fwhm=fwhm,
            gauss_frac=gauss_frac,
        )

        print(f"\n{pattern_name}")
        print(f"  Pearson: {_format_topk_profile(by_pearson, 'pearson', top_k_to_print)}")
        print(f"  Cosine:  {_format_topk_profile(by_cosine, 'cosine', top_k_to_print)}")

        # Step 3: visualize the best matches
        vis_pc.plot_summary(
            pattern_name,
            two_theta,
            exp_profile,
            by_pearson[0],
            by_cosine[0],
            simulated_profiles,
            show_plot=True,
            display_size=900
        )

This method compares full simulated and observed profiles, instead of matching only discrete peak positions.


In [ ]:
# @title Run Profile-Correlation
run_profile_correlation()

# Try on your own:
  # Increase/decrease broadening and see ranking stability.

# Examples below
  # run_profile_correlation(fwhm=0.20)
  # run_profile_correlation(fwhm=0.55)


## 2c) Sequential Rietveld-Style Refinement


In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

import importlib
from tutorial_utils.conventional import rietveld as rv
from tutorial_utils.sections import conventional_rietveld as vis_rv

# Force reload to get latest changes
sm = importlib.reload(rv)
vis_sm = importlib.reload(vis_rv)


def _format_topk_rietveld(rows, top_k):
    return ", ".join(
        f"{row['phase']} (Rwp={row['rwp']:.2f}, r={row['pearson']:.3f})"
        for row in rows[:top_k]
    )

"""
We will perform Rietveld refinement in a series of three steps:
background, then peak position, then peak width
"""

def run_background_refinement_step(two_theta, y_obs, structure,
        calculator, min_angle, max_angle, background_degree, fwhm_init):
    """Step 1: refine background with fixed lattice + fixed width."""
    return rv.refine_background_step(
        two_theta,
        y_obs,
        structure,
        calculator,
        min_angle=min_angle,
        max_angle=max_angle,
        background_degree=background_degree,
        fwhm_init=fwhm_init,
    )

def run_peak_position_refinement_step(two_theta, y_obs, structure,
        calculator, y_bg, min_angle, max_angle, fwhm_init):
    """Step 2: refine peak positions via lattice-scale refinement."""
    return rv.refine_lattice_step(
        two_theta,
        y_obs,
        structure,
        calculator,
        y_bg,
        min_angle=min_angle,
        max_angle=max_angle,
        fwhm_init=fwhm_init,
    )

def run_peak_width_refinement_step(two_theta, y_obs,
        peak_pos, peak_int, y_bg, fwhm_init):
    """Step 3: refine global peak width with lattice/background fixed."""
    return rv.refine_width_step(
        two_theta,
        y_obs,
        peak_pos,
        peak_int,
        y_bg,
        fwhm_init=fwhm_init,
    )

def assemble_rietveld_row(phase, y_obs, step1, step2, step3):
    """Collect key outputs from all three sequential refinement steps."""
    return {
        "phase": phase,
        "rwp": rv.compute_rwp(y_obs, step3["y_fit"]),
        "pearson": rv.pearson_corr(y_obs, step3["y_fit"]),
        "scales": step2["scales"],
        "fwhm": step3["fwhm"],
        "y_bg": step1["y_bg"],
        "y_fit_step1": step1["y_fit"],
        "y_fit_step2": step2["y_fit"],
        "y_fit_final": step3["y_fit"],
    }

def run_rietveld(
    top_k_to_print=3,                 # phases to show by Rwp
    patterns_to_run=("TiO2",),        # which patterns to refine
    background_degree=6,              # Chebyshev background degree
    fwhm_init=0.30,                   # initial peak width (deg)
):

    EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
    REFERENCE_DIR = Path("data/reference_structures")
    OUTPUT_DIR = Path("outputs/conventional/rietveld")
    MIN_ANGLE, MAX_ANGLE = 10.0, 80.0

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    vis_rv.OUTPUT_DIR = OUTPUT_DIR
    vis_rv.TOP_K_TO_PRINT = top_k_to_print

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    if patterns_to_run is not None:
        if isinstance(patterns_to_run, str):
            patterns_to_run = [patterns_to_run]
        keep = set(patterns_to_run)
        exp_files = [f for f in exp_files if f.stem in keep]

    # Step 0: load candidate structures and create calculator
    structures = rv.load_reference_structures(sorted(REFERENCE_DIR.glob("*.cif")))
    calculator = rv.make_calculator()

    all_rows = []
    print(f"Rietveld sequential | patterns={len(exp_files)} refs={len(structures)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: load and preprocess experimental profile
        two_theta, y_obs = rv.load_experimental_profile(
            exp_file,
            min_angle=MIN_ANGLE,
            max_angle=MAX_ANGLE,
        )

        # Step 2: run sequential refinement per candidate phase
        rows = []
        for phase, structure in structures.items():
            step1 = run_background_refinement_step(
                two_theta,
                y_obs,
                structure,
                calculator,
                min_angle=MIN_ANGLE,
                max_angle=MAX_ANGLE,
                background_degree=background_degree,
                fwhm_init=fwhm_init,
            )
            step2 = run_peak_position_refinement_step(
                two_theta,
                y_obs,
                structure,
                calculator,
                step1["y_bg"],
                min_angle=MIN_ANGLE,
                max_angle=MAX_ANGLE,
                fwhm_init=fwhm_init,
            )
            step3 = run_peak_width_refinement_step(
                two_theta,
                y_obs,
                step2["peak_pos"],
                step2["peak_int"],
                step1["y_bg"],
                fwhm_init=fwhm_init,
            )
            rows.append(assemble_rietveld_row(phase, y_obs, step1, step2, step3))

        # Step 3: rank by final fit quality
        rows.sort(key=lambda r: r["rwp"])

        print(f"\n{pattern_name}")
        print(f"  Rwp: {_format_topk_rietveld(rows, top_k_to_print)}")

        # Step 4: show each sequential step for the best phase
        vis_rv.plot_refinement_summary(
            pattern_name,
            two_theta,
            y_obs,
            rows[0],
            show_plot=True,
            display_size=900
        )

This simplified workflow refines background, lattice scales, and peak width in sequence for each candidate phase.


In [ ]:
# @title Run Sequential Rietveld Refinement
# This will take a few minutes!
run_rietveld(
    top_k_to_print=3,                 # phases shown by Rwp
    patterns_to_run=("TiO2",),        # patterns to process
    background_degree=6,              # background flexibility
    fwhm_init=0.30,                   # initial width guess
)

# Try on your own:
  # 1) Change background flexibility
  # 2) Change initial peak width

# Examples below
  # run_rietveld(background_degree=4)
  # run_rietveld(background_degree=8)
  # run_rietveld(fwhm_init=0.18)
  # run_rietveld(fwhm_init=0.45)


## Next Steps
Continue to **03 — ML + Deep Learning**: [Open in Colab](https://colab.research.google.com/github/Szymanski-Group/MRS_CH08_Tutorial/blob/main/notebooks/03_ML-Deep-Learning.ipynb)